# Neural Pattern Associator (NPA)

Paper: [Within-basket Recommendation via Neural Pattern Associator](https://arxiv.org/abs/2401.16433)

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import torch
import os

os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'
# os.environ['PYTORCH_MPS_HIGH_WATERMARK_RATIO'] = '0.0'

device = 'cpu'
print(f"Using device: {device}")

torch.set_num_threads(1)
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'

print(f"Torch threads: {torch.get_num_threads()}")

Using device: cpu
Torch threads: 1


In [3]:
import sys
import os

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
os.chdir(project_root)

from tecd_retail_recsys.data import DataPreprocessor
from tecd_retail_recsys.models.npa import NPATrainer


In [4]:
dp = DataPreprocessor(
    day_begin=1082, 
    day_end=1308, 
    val_days=20, 
    test_days=20, 
    min_user_interactions=1, 
    min_item_interactions=20
)
train_df, val_df, test_df = dp.preprocess()

Starting data preprocessing...
Loading events from t_ecd_small_partial/dataset/small/retail/events
Loaded 236,479,226 total events
Loading items data from t_ecd_small_partial/dataset/small/retail/items.pq
Loaded 250,171 items with features: ['item_id', 'item_brand_id', 'item_category', 'item_subcategory', 'item_price', 'item_embedding']
Merged item features. Data shape: (236479226, 12)
Filtered to 3,758,762 events with action_type='added-to-cart'
After filtering (min_user_interactions=1, min_item_interactions=20): 3,249,972 events, 84,944 users, 30,954 items
Created mappings: 84944 users, 30954 items
Temporal split - Train: days < 1269 (902,543 events), Val: days 1269-1288 (228,339 events), Test: days >= 1289 (223,395 events)
Users in each part (train, val, test) - 7425


In [ ]:
import pandas as pd
import numpy as np

def group_by_baskets(df, time_window_hours=2, min_basket_size=2):
    """
    Group interactions into baskets by user using time windows.
    """
    
    if not pd.api.types.is_datetime64_any_dtype(df['timestamp']):
        df = df.copy()
        df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s')
    
    df = df.sort_values(['user_id', 'timestamp']).reset_index(drop=True)
    
    user_baskets = {}
    
    for user_id, group in df.groupby('user_id'):
        group = group.sort_values('timestamp')
        timestamps = group['timestamp'].values
        items = group['item_id'].values
        time_delta = pd.to_timedelta(time_window_hours, unit='h')
        
        baskets = []
        current_basket = [items[0]]
        
        for i in range(1, len(items)):
            gap = pd.Timestamp(timestamps[i]) - pd.Timestamp(timestamps[i - 1])
            if gap <= time_delta:
                current_basket.append(items[i])
            else:
                if len(current_basket) >= min_basket_size:
                    seen = set()
                    deduped = []
                    for item in current_basket:
                        if item not in seen:
                            seen.add(item)
                            deduped.append(item)
                    baskets.append(deduped)
                current_basket = [items[i]]

        if len(current_basket) >= min_basket_size:
            seen = set()
            deduped = []
            for item in current_basket:
                if item not in seen:
                    seen.add(item)
                    deduped.append(item)
            baskets.append(deduped)
        
        if baskets:
            user_baskets[user_id] = baskets
    
    user_ids = list(user_baskets.keys())
    return user_baskets, user_ids


train_user_baskets, train_user_ids = group_by_baskets(train_df, time_window_hours=2, min_basket_size=2)
val_user_baskets, val_user_ids = group_by_baskets(val_df, time_window_hours=2, min_basket_size=2)
test_user_baskets, test_user_ids = group_by_baskets(test_df, time_window_hours=2, min_basket_size=2)

basket_sizes = [len(b) for baskets in train_user_baskets.values() for b in baskets]
seq_lengths = [len(baskets) for baskets in train_user_baskets.values()]

print(f"Users:                 {len(train_user_baskets)}")
print(f"Total baskets:         {len(basket_sizes)}")
print(f"Basket size:           median={np.median(basket_sizes):.0f}, "
      f"mean={np.mean(basket_sizes):.1f}, "
      f"max={np.max(basket_sizes)}")
print(f"Baskets per user:      median={np.median(seq_lengths):.0f}, "
      f"mean={np.mean(seq_lengths):.1f}, "
      f"max={np.max(seq_lengths)}")
print(f"Users with ≥2 baskets: {sum(1 for s in seq_lengths if s >= 2)} "
      f"({sum(1 for s in seq_lengths if s >= 2)/len(seq_lengths)*100:.0f}%)")


Users:                 7165
Total baskets:         119369
Basket size:           median=4, mean=6.4, max=117
Baskets per user:      median=11, mean=16.7, max=258
Users with ≥2 baskets: 6677 (93%)


In [12]:
trainer = NPATrainer(
    train_user_baskets,
    # === Architecture ===
    embed_dim=256,           # размерность эмбеддингов
    n_layers=4,              # кол-во слоёв VQA (stacked)
    n_channels=8,            # кол-во параллельных VQA каналов в слое
    codebook_size=64,        # размер кодбука паттернов
    dropout=0.1,
    variant="MC",            # "SC" (squashed-context) или "MC" (multi-context)
    # === Training ===
    lr=1e-3,
    weight_decay=1e-5,
    batch_size=256,
    epochs=20,
    # === Evaluation ===
    holdout_ratio=0.15,       # доля корзин для eval
    target_ratio=0.3,        # последние 20% корзины → ground truth
    eval_k=100,              # NDCG@100
    # === Other ===
    max_basket_size=30,
    seed=42,
    device="auto",           # "cpu", "cuda", "auto"
    verbose=True,
)

Items:         30696
Total baskets: 119369
Train baskets: 101464
Eval baskets:  17905
Device:        cpu
Model params:  31,290,345


In [13]:
trainer.train()

Epoch   1/20 │ loss=9.7076 │ NDCG@100=0.0415 │ Recall@100=0.1235
Epoch   2/20 │ loss=9.3742 │ NDCG@100=0.0455 │ Recall@100=0.1337
Epoch   3/20 │ loss=9.2398 │ NDCG@100=0.0515 │ Recall@100=0.1463
Epoch   4/20 │ loss=9.1207 │ NDCG@100=0.0579 │ Recall@100=0.1586
Epoch   5/20 │ loss=9.0062 │ NDCG@100=0.0619 │ Recall@100=0.1690
Epoch   6/20 │ loss=8.8743 │ NDCG@100=0.0643 │ Recall@100=0.1759
Epoch   7/20 │ loss=8.7651 │ NDCG@100=0.0667 │ Recall@100=0.1798
Epoch   8/20 │ loss=8.6367 │ NDCG@100=0.0682 │ Recall@100=0.1843
Epoch   9/20 │ loss=8.5162 │ NDCG@100=0.0696 │ Recall@100=0.1874
Epoch  10/20 │ loss=8.3827 │ NDCG@100=0.0704 │ Recall@100=0.1874
Epoch  11/20 │ loss=8.2557 │ NDCG@100=0.0709 │ Recall@100=0.1885
Epoch  12/20 │ loss=8.1288 │ NDCG@100=0.0704 │ Recall@100=0.1855
Epoch  13/20 │ loss=8.0116 │ NDCG@100=0.0707 │ Recall@100=0.1864
Epoch  14/20 │ loss=7.9164 │ NDCG@100=0.0699 │ Recall@100=0.1848
Epoch  15/20 │ loss=7.8247 │ NDCG@100=0.0705 │ Recall@100=0.1827
Epoch  16/20 │ loss=7.747

In [15]:
metrics = trainer.evaluate()

Eval  NDCG@100  = 0.0695
Eval  Recall@100 = 0.1806


In [16]:
model_path = "./models/npa/best_model.pt"

checkpoint = {
    "model_state_dict": trainer.model.state_dict(),
    "optimizer_state_dict": trainer.optimizer.state_dict(),
    "scheduler_state_dict": trainer.scheduler.state_dict(),
    "model_config": {
        "num_items": trainer.model.num_items,
        "embed_dim": trainer.model.embed_dim,
        "n_layers": len(trainer.model.layers),
        "n_channels": len(trainer.model.layers[0].channels),
        "codebook_size": trainer.model.layers[0].channels[0].codebook_size,
        "variant": trainer.model.variant,
    },
    "trainer_config": {
        "eval_k": trainer.eval_k,
        "target_ratio": trainer.target_ratio,
        "batch_size": trainer.batch_size,
    },
}
torch.save(checkpoint, model_path)

In [17]:
# инференс для конкретной корзины (аля realtime)
basket = [8591, 4010, 750]
recommendations = trainer.predict(basket, top_k=10)
print("\nTop-10 recommendations:")
for item_id, score in recommendations:
    print(f"  item {item_id:>6d}  score={score:.4f}")


Top-10 recommendations:
  item  23428  score=0.0037
  item  18083  score=0.0033
  item  30138  score=0.0026
  item   2093  score=0.0023
  item   3805  score=0.0022
  item   4585  score=0.0021
  item  14876  score=0.0020
  item   3764  score=0.0020
  item  17934  score=0.0020
  item   9364  score=0.0018


In [18]:
# разные стратегии рекомендаций на подвыборке пользователей

sample_val_user_baskets = {k:v for k,v in val_user_baskets.items() if k in list(val_user_baskets.keys())[:100]}

# Greedy
res_greedy = trainer.evaluate_autoregressive(
    sample_val_user_baskets,
    target_ratio=0.3,
    top_k=100,
    strategy="greedy",
)

# Nucleus sampling
res_nucleus = trainer.evaluate_autoregressive(
    sample_val_user_baskets,
    target_ratio=0.3,
    top_k=100,
    strategy="nucleus",
    top_p=0.9,
    temperature=0.7,
)
# Multirun top-k sampling
res_multi = trainer.evaluate_autoregressive(
    sample_val_user_baskets,
    target_ratio=0.3,
    top_k=100,
    strategy="top_k",
    top_k_sample=200,
    temperature=0.7,
    n_runs=10,
)

# Default (not autoregressive) sampling
res_single = trainer.evaluate_external(sample_val_user_baskets, target_ratio=0.3, top_k=100)

print("\n===== Сравнение =====")
print(f"Default (Not-AR):  NDCG={res_single['ndcg']:.4f}  Recall={res_single['recall']:.4f}")
print(f"AR greedy:         NDCG={res_greedy['ndcg']:.4f}  Recall={res_greedy['recall']:.4f}")
print(f"AR nucleus:        NDCG={res_nucleus['ndcg']:.4f}  Recall={res_nucleus['recall']:.4f}")
print(f"AR multi-run (10): NDCG={res_multi['ndcg']:.4f}  Recall={res_multi['recall']:.4f}")


Strategy:         greedy
Baskets evaluated: 314
NDCG@100:       0.0504
Recall@100:     0.1370
HitRate@100:    0.2643
Strategy:         nucleus
Baskets evaluated: 314
NDCG@100:       0.0241
Recall@100:     0.0834
HitRate@100:    0.1720
Strategy:         top_k (x10 runs)
Baskets evaluated: 314
NDCG@100:       0.0429
Recall@100:     0.1378
HitRate@100:    0.2643
Baskets evaluated: 314
NDCG@100:       0.0510
Recall@100:     0.1450

===== Сравнение =====
Default (Not-AR):  NDCG=0.0510  Recall=0.1450
AR greedy:         NDCG=0.0504  Recall=0.1370
AR nucleus:        NDCG=0.0241  Recall=0.0834
AR multi-run (10): NDCG=0.0429  Recall=0.1378


In [19]:
# evaluation with best strategy
best_strategy_metrics = trainer.evaluate_external(
    val_user_baskets,
    target_ratio=0.3,
    top_k=100
)

Baskets evaluated: 21263
NDCG@100:       0.0595
Recall@100:     0.1559


`Удалось добиться ndcg@100 = 0.0595 на задаче within-basket recommendation. Результат ниже, чем у персонализированных подходов, использующих историю покупок. Модель NPA не требует истории пользователя и опирается исключительно на паттерны совместных покупок, выученные из обучающих корзин. Это делает ее применимой в сценарии холодного старта, где персонализированные методы неработоспособны. Обученная модель работает в real-time режиме, адаптируя рекомендации по мере добавления товаров в корзину. Ограничение: модель не учитывает индивидуальные предпочтения пользователя и не способна рекомендовать товары, отсутствующие в обучающей выборке.`